# ideal-cnt-cb-network — the anatomy of *one* run

This notebook walks a **single** Monte Carlo run end to end, calling the
`cntcb` kernel and engines directly, to show what the machinery does before
you unleash the parameter sweeps in `generate/`. It is a mechanism tour,
**not** a campaign: tiny RVE, a handful of realizations, seconds to run.
Three parts — percolation, conductivity, gauge factor.

If you just want first numbers from the command line, start with
`examples/quickstart.md` instead; come back here when you want to see the
engine underneath.

## 0. Setup

From the repository root, `pip install -e .` (then the imports below just work),
or run Jupyter with the repo root on `PYTHONPATH`. Requires `numpy` and `scipy`.

In [1]:
from cntcb.kernel.materials import carbon_black

cb = carbon_black(diameter_nm=80.0)
cb.name, cb.diameter_nm, cb.aspect_ratio

('Carbon Black', 80.0, 1.0)

## 1. Percolation — one draw is noisy; the answer is an ensemble

`find_threshold_cb` searches for the volume fraction at which the CB network
first spans the box. Run it with a **single** realization and then with a few:
the single draw scatters, and the reportable number is the ensemble
`mean ± std`. That is the whole reason `generate/` averages over `--n-real`.

In [2]:
from cntcb.kernel.percolation_finder import find_threshold_cb

one = find_threshold_cb(cb, rve_size=800.0, n_max=900, n_realizations=1, seed=1, verbose=False)
ens = find_threshold_cb(cb, rve_size=800.0, n_max=900, n_realizations=4, seed=1, verbose=False)

print(f"one draw    : phi_c = {one.phi_c_mean:.4f}")
print(f"ensemble(4) : phi_c = {ens.phi_c_mean:.4f} +/- {ens.phi_c_std:.4f}"
      f"   (analytical {ens.phi_c_analytical:.4f})")

one draw    : phi_c = 0.2288
ensemble(4) : phi_c = 0.2234 +/- 0.0247   (analytical 0.2401)


## 2. Conductivity — one realization, inspected

Now weight each accepted junction by its Simmons tunneling conductance and solve
the resistor network **once**. The result object carries not just `sigma_s_per_m`
but the graph diagnostics (filler and edge counts) so you can see the network you
built. We place CB only here (`phi_cnt = 0`).

In [3]:
from cntcb.engines.percolation_engine import (
    estimate_cb_n_max, estimate_cnt_n_max, solve_truncated_lognormal,
)
from cntcb.engines.simmons_conductivity_engine import run_simmons_conductivity_realization

cb_model  = solve_truncated_lognormal(148.0, 83.0, 20.0, 500.0)
cnt_model = solve_truncated_lognormal(500.0, 300.0, 50.0, 1500.0)
phi_cb = 0.35

n_cb  = estimate_cb_n_max(cb_model, phi_cb, 1000.0, safety=2.5)
n_cnt = estimate_cnt_n_max(cnt_model, 1e-12, 1000.0, 10.0, safety=2.5)

res = run_simmons_conductivity_realization(
    cb_model=cb_model, cnt_model=cnt_model,
    phi_cb_target_vol=phi_cb, phi_cnt_target_vol=0.0,
    l_rve_nm=1000.0, cnt_diam_nm=10.0, waviness=0.7, tunnel_nm=10.0,
    seg_len_unit_nm=50.0, d_min_nm=0.34, g_cutoff_s=1e-15,
    seed=7, n_cb_max=n_cb, n_cnt_max=n_cnt,
)
print(f"success = {res.success}   sigma = {res.sigma_s_per_m:.4g} S/m")
print(f"n_cb = {res.n_cb}   n_edges = {res.n_edges}")

success = True   sigma = 652.8 S/m
n_cb = 81   n_edges = 70


## 3. Gauge factor — the ideal-affine reference

Deform the same network affinely over a small strain grid and fit
`d ln R / d eps`. With the junction-opening coefficients at zero the network is
purely **affine**: the conductivity term is negative and offsets the geometric
term `1 + 2*nu ≈ 1.72`. In this CB-only single realization the cancellation is
essentially exact (GF ≈ 0); in the CNT-bearing production ensembles (`n = 300`,
`L_RVE = 4000 nm`) the ideal-affine reference GF is about 0.25–0.27 — still far
below the geometric value. Turn on a non-affine sphere–sphere opening and the
GF lifts to a measurable value. **That is the paper's point:** measured GF above
the ideal-affine reference is interpreted as evidence for non-affine junction
opening.

In [4]:
from cntcb.engines.gf_engine import run_simmons_gf_realization

def one_gf(alpha_ss):
    return run_simmons_gf_realization(
        cb_model=cb_model, cnt_model=cnt_model,
        phi_cb_target_vol=phi_cb, phi_cnt_target_vol=0.0,
        l_rve_nm=1000.0, cnt_diam_nm=10.0, waviness=0.7, tunnel_nm=10.0,
        seg_len_unit_nm=50.0, d_min_nm=0.34, g_cutoff_s=1e-15,
        strain_grid=(0.0, 0.0025, 0.005, 0.0075, 0.01), poisson_nu=0.36,
        seed=7, opening_alpha_ss_nm_per_strain=alpha_ss,
        n_cb_max=n_cb, n_cnt_max=n_cnt,
    )

ideal  = one_gf(0.0)
nonaff = one_gf(200.0)
print(f"ideal    (alpha=0)   : GF_r = {ideal.gf_r:.4g}"
      f"   [gf_sigma {ideal.gf_sigma:.3f} + gf_geom {ideal.gf_geom:.3f}]")
print(f"non-affine(alpha=200): GF_r = {nonaff.gf_r:.4g}")

ideal    (alpha=0)   : GF_r = 3.632e-12   [gf_sigma -1.716 + gf_geom 1.716]
non-affine(alpha=200): GF_r = 0.4221


### Where to go next

- First numbers in five minutes, from the command line:
  `examples/quickstart.md`.
- Sweep your own parameters: `generate/percolation_threshold.py`,
  `generate/conductivity.py`, `generate/gauge_factor.py`.
- Put *measured* literature geometry through the tools end to end:
  `examples/pan2013_case_study.md`.
- Reproduce a paper figure exactly: `reproduce_paper/`.
- The full method and its interpretation are in the accompanying paper.